# LightECGNet v2

---
## Section 1 — Imports & GPU setup


In [ ]:
import os, csv, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from scipy.io import loadmat
from scipy.signal import butter, filtfilt
from collections import Counter
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, accuracy_score
)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# RTX 40xx optimisations
torch.backends.cudnn.benchmark       = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

AMP_DTYPE   = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_AMP     = device.type == 'cuda'

NUM_WORKERS = 0
PERSISTENT_WORKERS = False

SIGNAL_LEN  = 5000  # 10 seconds at 500 Hz 
print(f'AMP dtype   : {AMP_DTYPE}')
print(f'Signal len  : {SIGNAL_LEN}')
print(f'num_workers : {NUM_WORKERS}')

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU | VRAM: 8.6 GB
AMP dtype   : torch.bfloat16
Signal len  : 5000
num_workers : 0


---
## Section 2 — Data loading & preprocessing


In [ ]:
snomed_to_acronym = {}
with open('ConditionNames_SNOMED-CT.csv', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        snomed_to_acronym[row['Snomed_CT'].strip()] = row['Acronym Name'].strip()
print('SNOMED mapping:', len(snomed_to_acronym), 'entries')


SNOMED mapping: 55 entries


In [ ]:
def extract_labels(hea_path):
    with open(hea_path, 'r') as f:
        for line in f:
            if '#Dx' in line.replace(' ', ''):
                return line.split(':')[1].strip().split(',')
    return []

def bandpass_filter(signal, low=0.5, high=40.0, fs=500, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, signal)

def load_ecg_mat(path, target_len=5000):
    """Load ECG .mat file and pad/crop to target_len (paper: 5000 = 10s @ 500Hz)."""
    mat    = loadmat(path)
    signal = mat.get('val', mat.get('data'))
    if signal is None:
        raise ValueError(f'No ECG data in {path}')
    if signal.shape[0] != 12:
        signal = signal.T

    current_len = signal.shape[1]
    if current_len < target_len:
        pad = np.zeros((12, target_len - current_len), dtype=signal.dtype)
        signal = np.concatenate([signal, pad], axis=1)
    elif current_len > target_len:
        signal = signal[:, :target_len]

    for i in range(12):
        signal[i] = bandpass_filter(signal[i])

    # Per-lead z-score normalization
    signal = (signal - signal.mean(axis=1, keepdims=True)) / \
             (signal.std(axis=1, keepdims=True) + 1e-8)
    return signal.astype(np.float32)

def augment_ecg(signal):
    """Training augmentation: noise, shift, scale, lead dropout."""
    sig = signal.copy()
    if np.random.rand() < 0.5:
        sig = sig + np.random.normal(0, 0.02, sig.shape).astype(np.float32)
    if np.random.rand() < 0.5:
        sig = np.roll(sig, np.random.randint(-250, 250), axis=1)
    if np.random.rand() < 0.5:
        sig = sig * float(np.random.uniform(0.8, 1.2))
    if np.random.rand() < 0.3:
        sig[np.random.randint(0, 12)] = 0.0
    return sig

def tta_augment(signal):
    sig = signal.copy()
    sig = sig + np.random.normal(0, 0.005, sig.shape).astype(np.float32)
    sig = sig * float(np.random.uniform(0.95, 1.05))
    return sig


In [5]:
class ECGDataset(Dataset):
    EXCLUDE = {'ABI', 'VET', 'FQRS', 'SAAWR', 'JPT', 'VB'}

    def __init__(self, root_dir):
        records, labels_list = [], []
        for root, _, files in os.walk(root_dir):
            for fname in sorted(files):
                if not fname.endswith('.hea'):
                    continue
                base   = os.path.join(root, fname[:-4])
                codes  = extract_labels(base + '.hea')
                labels = [snomed_to_acronym[c] for c in codes if c in snomed_to_acronym]
                if labels:
                    records.append(base)
                    labels_list.append(labels)

        counts            = Counter(lbl for sub in labels_list for lbl in sub)
        self.classes      = sorted(c for c in counts if c not in self.EXCLUDE)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.records, self.labels_list = [], []
        for r, lbls in zip(records, labels_list):
            valid = [l for l in lbls if l in self.class_to_idx]
            if valid:
                self.records.append(r)
                self.labels_list.append(valid)

        print(f'Classes : {len(self.classes)}')
        print(f'Records : {len(self.records)}')
        print('Preloading signals into RAM (as tensors for zero-copy DataLoader)...')

        self.signals, self.targets = [], []
        skipped = 0
        for rec, lbls in zip(self.records, self.labels_list):
            try:
                sig = load_ecg_mat(rec + '.mat', target_len=SIGNAL_LEN)
                y   = np.zeros(len(self.classes), dtype=np.float32)
                for lbl in lbls:
                    y[self.class_to_idx[lbl]] = 1.0
                # SPEED: Store as torch tensors — avoids np->tensor conversion per batch
                self.signals.append(torch.from_numpy(sig))
                self.targets.append(torch.from_numpy(y))
            except Exception as e:
                skipped += 1

        if skipped > 0:
            print(f'Skipped {skipped} records due to loading errors')

        # Compute positive weights for BCEWithLogitsLoss
        all_targets = torch.stack(self.targets).numpy()
        pos_counts  = all_targets.sum(axis=0)
        neg_counts  = len(all_targets) - pos_counts
        self.pos_weight = np.clip(neg_counts / (pos_counts + 1), 1.0, 50.0)
        print(f'Pos weight range: [{self.pos_weight.min():.1f}, {self.pos_weight.max():.1f}]')

        self._mode = 'clean'
        print(f'Dataset ready: {len(self.signals)} records loaded.')

    def train_mode(self): self._mode = 'train'
    def eval_mode(self):  self._mode = 'clean'
    def tta_mode(self):   self._mode = 'tta'

    def __len__(self): return len(self.signals)

    def __getitem__(self, idx):
        sig = self.signals[idx]
        y   = self.targets[idx]
        if self._mode == 'train':
            sig = torch.from_numpy(augment_ecg(sig.numpy()))
        elif self._mode == 'tta':
            sig = torch.from_numpy(tta_augment(sig.numpy()))
        return sig, y


dataset     = ECGDataset('dataset/WFDBRecords')
NUM_CLASSES = len(dataset.classes)
print('Num classes:', NUM_CLASSES)

Classes : 45
Records : 44476
Preloading signals into RAM (as tensors for zero-copy DataLoader)...
Pos weight range: [1.7, 50.0]
Dataset ready: 44476 records loaded.
Num classes: 45


In [6]:
rng       = np.random.RandomState(42)
indices   = rng.permutation(len(dataset))
folds     = np.array_split(indices, 10)
TEST_IDX  = folds[9]   # held-out test fold — never trained on
NUM_FOLDS = 9
os.makedirs('models', exist_ok=True)
print(f'Test set : {len(TEST_IDX):,} samples')
print(f'CV pool  : {len(dataset)-len(TEST_IDX):,} samples over {NUM_FOLDS} folds')


Test set : 4,447 samples
CV pool  : 40,029 samples over 9 folds


---
## Section 3 — Model architecture

Key design: LightECGNet uses depthwise-separable convolutions and dilated TCN
blocks.



In [7]:
class DSConv1d(nn.Module):
    """Depthwise-separable 1D conv: DW -> PW -> BN -> act."""
    def __init__(self, in_ch, out_ch, kernel, dilation=1, act='relu'):
        super().__init__()
        pad      = (kernel - 1) * dilation // 2
        self.dw  = nn.Conv1d(in_ch, in_ch, kernel, padding=pad,
                             dilation=dilation, groups=in_ch, bias=False)
        self.pw  = nn.Conv1d(in_ch, out_ch, 1, bias=False)
        self.bn  = nn.BatchNorm1d(out_ch)
        if act == 'prelu':
            self.act = nn.PReLU()
        else:
            self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class SEBlock1d(nn.Module):
    """Squeeze-and-Excitation channel attention (paper's GCAB uses similar concept)."""
    def __init__(self, ch, reduction=8):
        super().__init__()
        mid       = max(ch // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1  = nn.Linear(ch, mid)
        self.fc2  = nn.Linear(mid, ch)
    def forward(self, x):
        s = self.pool(x).squeeze(-1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))
        return x * s.unsqueeze(-1)


class LeadAttention(nn.Module):
    """Per-sample attention over 12 ECG leads."""
    def __init__(self, n_leads=12, reduction=2):
        super().__init__()
        mid       = max(n_leads // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1  = nn.Linear(n_leads, mid)
        self.fc2  = nn.Linear(mid, n_leads)
    def forward(self, x):
        s = self.pool(x).squeeze(-1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))
        return x * s.unsqueeze(-1)
    @torch.no_grad()
    def get_weights(self, x):
        s = self.pool(x).squeeze(-1)
        return torch.sigmoid(self.fc2(F.relu(self.fc1(s))))


class DilatedTCNBlock(nn.Module):
    """Dilated TCN Residual Block (inspired by paper's SRB).
    DW dilated conv -> BN -> ReLU -> PW conv -> BN -> ReLU -> SE -> skip."""
    def __init__(self, in_ch, out_ch, kernel=9, dilation=1, se_reduction=8, drop=0.15):
        super().__init__()
        pad       = (kernel - 1) * dilation // 2
        self.dw   = nn.Conv1d(in_ch, in_ch, kernel, padding=pad,
                              dilation=dilation, groups=in_ch, bias=False)
        self.bn1  = nn.BatchNorm1d(in_ch)
        self.pw   = nn.Conv1d(in_ch, out_ch, 1, bias=False)
        self.bn2  = nn.BatchNorm1d(out_ch)
        self.se   = SEBlock1d(out_ch, reduction=se_reduction)
        self.drop = nn.Dropout(drop)
        self.relu = nn.ReLU(inplace=True)
        self.skip = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        r = self.skip(x)
        x = self.relu(self.bn1(self.dw(x)))
        x = self.relu(self.bn2(self.pw(x)))
        x = self.se(x)
        x = self.drop(x)
        return self.relu(x + r)


In [ ]:
class LightECGNetV2(nn.Module):
    def __init__(self, num_classes, n_leads=12):
        super().__init__()

        self.lead_attn = LeadAttention(n_leads, reduction=2)

        # Multi-scale DS stem
        self.stem_k3   = DSConv1d(n_leads, 16, kernel=3)
        self.stem_k7   = DSConv1d(n_leads, 16, kernel=7)
        self.stem_k15  = DSConv1d(n_leads, 32, kernel=15)
        self.stem_proj = nn.Sequential(
            nn.Conv1d(64, 64, 1, bias=False),
            nn.BatchNorm1d(64),
            nn.PReLU()
        )
        self.stem_pool = nn.AvgPool1d(2)   # 5000->2500

        # TCN Stage 1: local wave shape features (k=13 matches paper's SRB kernel)
        self.tcn1a = DilatedTCNBlock(64, 64, kernel=13, dilation=1, drop=0.1)
        self.tcn1b = DilatedTCNBlock(64, 64, kernel=13, dilation=2, drop=0.1)
        self.pool1 = nn.AvgPool1d(2)       # 2500->1250

        # TCN Stage 2: full-beat coverage
        self.tcn2a = DilatedTCNBlock(64,  128, kernel=13, dilation=4, drop=0.2)
        self.tcn2b = DilatedTCNBlock(128, 128, kernel=13, dilation=8, drop=0.2)
        self.pool2 = nn.AvgPool1d(2)       # 1250->625

        # TCN Stage 3: high-level rhythm + interval features
        self.tcn3a = DilatedTCNBlock(128, 256, kernel=9, dilation=1, drop=0.3)
        self.tcn3b = DilatedTCNBlock(256, 256, kernel=9, dilation=2, drop=0.3)

        # GMP+GAP -> 512 features -> FC head
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x   = self.lead_attn(x)
        x   = torch.cat([self.stem_k3(x), self.stem_k7(x), self.stem_k15(x)], dim=1)
        x   = self.stem_pool(self.stem_proj(x))
        x   = self.tcn1b(self.tcn1a(x))
        x   = self.pool1(x)
        x   = self.tcn2b(self.tcn2a(x))
        x   = self.pool2(x)
        x   = self.tcn3b(self.tcn3a(x))
        gmp = x.max(dim=-1).values
        gap = x.mean(dim=-1)
        logits = self.head(torch.cat([gmp, gap], dim=1))
        return logits


# Verify
_m = LightECGNetV2(NUM_CLASSES).to(device)
n  = sum(p.numel() for p in _m.parameters())
print(f'Parameters : {n:,}  |  {n*4/1e6:.2f} MB (FP32)')
with torch.no_grad():
    _o = _m(torch.randn(2, 12, SIGNAL_LEN, device=device))
print(f'Output     : {tuple(_o.shape)}  (expected: (2, {NUM_CLASSES}))')
del _m, _o
torch.cuda.empty_cache()

Parameters : 376,108  |  1.50 MB (FP32)
Output     : (2, 45)  (expected: (2, 45))


---
## Section 4 — Loss function


BCEWithLogitsLoss is numerically stable (log-sum-exp trick internally) and
pos_weight handles class imbalance effectively.


In [ ]:
# Primary loss: BCEWithLogitsLoss with class-specific pos_weight
pos_weight_tensor = torch.tensor(dataset.pos_weight, dtype=torch.float32).to(device)
print(f'Using BCEWithLogitsLoss with pos_weight (range [{pos_weight_tensor.min():.1f}, {pos_weight_tensor.max():.1f}])')


class DistillationLoss(nn.Module):
    """KD loss: alpha * BCE(hard) + (1-alpha) * T^2 * KL(soft)."""
    def __init__(self, alpha=0.5, temperature=3.0):
        super().__init__()
        self.alpha = alpha
        self.T     = temperature

    def forward(self, s_logits, t_logits, targets, bce_loss_fn):
        # Hard loss: standard BCE
        hard = bce_loss_fn(s_logits.float(), targets.float())

        # Soft loss: match teacher's soft predictions
        s = torch.sigmoid(s_logits.float() / self.T).clamp(1e-7, 1 - 1e-7)
        t = torch.sigmoid(t_logits.float() / self.T).clamp(1e-7, 1 - 1e-7).detach()
        soft = -(t * torch.log(s) + (1 - t) * torch.log(1 - s)).mean()

        return self.alpha * hard + (1 - self.alpha) * (self.T ** 2) * soft


Using BCEWithLogitsLoss with pos_weight (range [1.7, 50.0])


---
## Section 5 — Training utilities


In [10]:
def mixup_batch(x, y, alpha=0.2):
    """Batch-level ECG Mixup. Lambda >= 0.5 so primary sample dominates."""
    if alpha <= 0:
        return x, y
    lam  = float(max(np.random.beta(alpha, alpha),
                     1.0 - np.random.beta(alpha, alpha)))
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], lam * y + (1 - lam) * y[perm]


def find_optimal_thresholds(probs, trues, num_classes):
    """Find per-class optimal thresholds on validation set."""
    thresholds = np.full(num_classes, 0.5)
    for c in range(num_classes):
        if trues[:, c].sum() == 0:
            continue
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.1, 0.9, 0.05):
            preds_c = (probs[:, c] > t).astype(int)
            f1_c = f1_score(trues[:, c], preds_c, zero_division=0)
            if f1_c > best_f1:
                best_f1 = f1_c
                best_t  = t
        thresholds[c] = best_t
    return thresholds


def evaluate_model(model, loader, dataset_obj, n_tta=1, thresholds=None):
    """Inference with optional TTA.
    Returns probs (N,C), preds (N,C), trues (N,C).
    Uses per-class thresholds if provided, otherwise 0.5."""
    model.eval()
    tta_probs, trues = [], None

    for pass_idx in range(n_tta):
        dataset_obj.tta_mode() if pass_idx > 0 else dataset_obj.eval_mode()
        batch_p, batch_t = [], []

        with torch.no_grad():
            for x, y in loader:
                x = x.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                    logits = model(x)
                probs = torch.sigmoid(logits.float()).cpu().numpy()
                batch_p.append(probs)
                if pass_idx == 0:
                    batch_t.append(y.numpy())

        tta_probs.append(np.vstack(batch_p))
        if pass_idx == 0:
            trues = np.vstack(batch_t)

    dataset_obj.eval_mode()
    probs = np.mean(tta_probs, axis=0)

    if thresholds is not None:
        preds = (probs > thresholds[np.newaxis, :]).astype(int)
    else:
        preds = (probs > 0.5).astype(int)

    return probs, preds, trues


def compute_metrics(y_true, y_pred, y_prob, label='', verbose=True):
    mf1 = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    uf1 = f1_score(y_true, y_pred, average='micro',  zero_division=0)
    pre = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro',    zero_division=0)

    # Accuracy: use subset accuracy for multi-label (exact match per sample)
    # But also compute per-class average like the paper
    per_class_acc = []
    for c in range(y_true.shape[1]):
        correct = (y_pred[:, c] == y_true[:, c]).sum()
        per_class_acc.append(correct / len(y_true))
    acc = np.mean(per_class_acc)

    v   = y_true.sum(axis=0) > 0
    auc = roc_auc_score(y_true[:,v], y_prob[:,v],
                        average='macro', multi_class='ovr') \
          if v.sum() > 1 else float('nan')
    if verbose:
        b = lambda val, ref: '  BEAT' if val > ref else ''
        sep = '=' * 58
        print(f'\n{sep}\n {label}\n{sep}')
        print(f'  Precision (macro): {pre:.4f}   [paper: 0.394]{b(pre,0.394)}')
        print(f'  Recall    (macro): {rec:.4f}   [paper: 0.533]{b(rec,0.533)}')
        print(f'  F1        (macro): {mf1:.4f}   [paper: 0.413]{b(mf1,0.413)}')
        print(f'  F1        (micro): {uf1:.4f}')
        print(f'  AUROC     (macro): {auc:.4f}   [paper: 0.962]{b(auc,0.962)}')
        print(f'  Accuracy  (mean) : {acc:.4f}   [paper: 0.956]{b(acc,0.956)}')
    return dict(macro_f1=mf1, micro_f1=uf1, precision=pre,
                recall=rec, auroc=auc, accuracy=acc)


---
## Section 6 — Sanity check


In [11]:
print('Running sanity check — 3 mini-batches...')
_bce  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
_m    = LightECGNetV2(NUM_CLASSES).to(device)
_m.train()
_opt  = torch.optim.AdamW(_m.parameters(), lr=1e-3)
_subset = Subset(dataset, np.arange(min(32, len(dataset))))
_ldr    = DataLoader(_subset, batch_size=8, shuffle=True, num_workers=0)
dataset.train_mode()

losses = []
for i, (x, y) in enumerate(_ldr):
    if i >= 3: break
    x, y = x.to(device), y.to(device)
    _opt.zero_grad(set_to_none=True)

    with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = _m(x)

    # Loss in FP32
    loss = _bce(logits.float(), y.float())
    loss.backward()

    # Check gradient norms
    total_norm = 0
    for p in _m.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    total_norm = total_norm ** 0.5

    _opt.step()
    losses.append(loss.item())
    print(f'  Batch {i+1}: loss={loss.item():.6f}  grad_norm={total_norm:.4f}')

dataset.eval_mode()
del _m, _opt, _bce, _ldr, _subset
torch.cuda.empty_cache()

if all(l < 1e-6 for l in losses):
    print('\nFAIL — loss near zero! Something is still wrong.')
elif any(np.isnan(l) or np.isinf(l) for l in losses):
    print('\nFAIL — NaN/Inf in loss.')
else:
    print(f'\nSANITY CHECK PASSED — losses: {[f"{l:.4f}" for l in losses]}')
    print('Gradients are flowing. Proceed to Phase 1.')


Running sanity check — 3 mini-batches...
  Batch 1: loss=0.998376  grad_norm=5.8409
  Batch 2: loss=1.090812  grad_norm=5.9663
  Batch 3: loss=1.172630  grad_norm=6.2324

SANITY CHECK PASSED — losses: ['0.9984', '1.0908', '1.1726']
Gradients are flowing. Proceed to Phase 1.


---
## Section 7 — Phase 1: Standalone training


In [ ]:
NUM_EPOCHS_P1 = 120
BATCH_SIZE    = 256  
MIXUP_ALPHA   = 0.2
EVAL_EVERY    = 5
PATIENCE      = 15

bce_loss_fn     = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
fold_results_p1 = []

USE_COMPILE = False

for fold in range(NUM_FOLDS):
    print(f'\n{"#"*62}\n  PHASE 1 — FOLD {fold+1}/{NUM_FOLDS}\n{"#"*62}')

    val_idx = folds[fold]
    tr_idx  = np.concatenate([folds[i] for i in range(NUM_FOLDS) if i != fold])

    train_loader = DataLoader(
        Subset(dataset, tr_idx), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx), batch_size=BATCH_SIZE * 2,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
        
    )

    model     = LightECGNetV2(NUM_CLASSES).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    warmup    = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=5
    )
    cosine    = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS_P1 - 5, eta_min=1e-5
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[5]
    )

    best_f1    = 0.0
    best_ckpt  = f'models/lightv2_p1_fold{fold}.pt'
    best_thresh = None
    no_improve = 0

    for epoch in range(NUM_EPOCHS_P1):
        model.train()
        dataset.train_mode()
        total_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            x, y = mixup_batch(x, y, alpha=MIXUP_ALPHA)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(x)

            loss = bce_loss_fn(logits.float(), y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % EVAL_EVERY == 0 or epoch == NUM_EPOCHS_P1 - 1:
            dataset.eval_mode()
            probs, preds_default, trues = evaluate_model(
                model, val_loader, dataset, n_tta=1
            )

            thresholds = find_optimal_thresholds(probs, trues, NUM_CLASSES)
            preds_opt  = (probs > thresholds[np.newaxis, :]).astype(int)

            m_default = compute_metrics(trues, preds_default, probs, verbose=False)
            m_opt     = compute_metrics(trues, preds_opt, probs, verbose=False)

            if m_opt['macro_f1'] >= m_default['macro_f1']:
                f1 = m_opt['macro_f1']
                use_thresh = thresholds
            else:
                f1 = m_default['macro_f1']
                use_thresh = np.full(NUM_CLASSES, 0.5)

            lr = optimizer.param_groups[0]['lr']
            avg_loss = total_loss / len(train_loader)

            print(f'  Ep {epoch+1:3d}/{NUM_EPOCHS_P1}'
                  f'  loss={avg_loss:.4f}'
                  f'  val_F1={f1:.4f}'
                  f'  lr={lr:.2e}'
                  f'  patience={no_improve}/{PATIENCE}')

            if f1 > best_f1:
                best_f1     = f1
                best_thresh = use_thresh.copy()
                no_improve  = 0
                torch.save(model.state_dict(), best_ckpt)
                np.save(best_ckpt.replace('.pt', '_thresh.npy'), best_thresh)
                print(f'  -> Saved (F1={best_f1:.4f})')
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f'  -> Early stop at epoch {epoch+1}'
                          f'  (best F1={best_f1:.4f})')
                    break

    print(f'\n  Fold {fold+1} best val F1: {best_f1:.4f}')
    fold_results_p1.append({'fold': fold, 'best_f1': best_f1,
                             'ckpt': best_ckpt, 'val_idx': val_idx,
                             'thresholds': best_thresh})

avg_f1_p1 = float(np.mean([r['best_f1'] for r in fold_results_p1]))
print(f'\n{"="*62}\n  Phase 1 complete. Mean val F1 = {avg_f1_p1:.4f}\n{"="*62}')


##############################################################
  PHASE 1 — FOLD 1/9
##############################################################


D:\VS Code\python\material\jupyter\ECG\ECG_classification\ecg_gpu\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  Ep   5/120  loss=0.5458  val_F1=0.3003  lr=1.00e-03  patience=0/15
  -> Saved (F1=0.3003)
  Ep  10/120  loss=0.4609  val_F1=0.3703  lr=9.95e-04  patience=0/15
  -> Saved (F1=0.3703)
  Ep  15/120  loss=0.4150  val_F1=0.3948  lr=9.82e-04  patience=0/15
  -> Saved (F1=0.3948)
  Ep  20/120  loss=0.3891  val_F1=0.4134  lr=9.59e-04  patience=0/15
  -> Saved (F1=0.4134)
  Ep  25/120  loss=0.3812  val_F1=0.4301  lr=9.28e-04  patience=0/15
  -> Saved (F1=0.4301)
  Ep  30/120  loss=0.3481  val_F1=0.4395  lr=8.89e-04  patience=0/15
  -> Saved (F1=0.4395)
  Ep  35/120  loss=0.3501  val_F1=0.4399  lr=8.43e-04  patience=0/15
  -> Saved (F1=0.4399)
  Ep  40/120  loss=0.3435  val_F1=0.4322  lr=7.90e-04  patience=0/15
  Ep  45/120  loss=0.3267  val_F1=0.4402  lr=7.33e-04  patience=1/15
  -> Saved (F1=0.4402)
  Ep  50/120  loss=0.3339  val_F1=0.4450  lr=6.71e-04  patience=0/15
  -> Saved (F1=0.4450)
  Ep  55/120  loss=0.3221  val_F1=0.4407  lr=6.06e-04  patience=0/15
  Ep  60/120  loss=0.3295  val_F1=

D:\VS Code\python\material\jupyter\ECG\ECG_classification\ecg_gpu\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  Ep   5/120  loss=0.5335  val_F1=0.2914  lr=1.00e-03  patience=0/15
  -> Saved (F1=0.2914)
  Ep  10/120  loss=0.4504  val_F1=0.3699  lr=9.95e-04  patience=0/15
  -> Saved (F1=0.3699)
  Ep  15/120  loss=0.4281  val_F1=0.3842  lr=9.82e-04  patience=0/15
  -> Saved (F1=0.3842)
  Ep  20/120  loss=0.3894  val_F1=0.4006  lr=9.59e-04  patience=0/15
  -> Saved (F1=0.4006)
  Ep  25/120  loss=0.3784  val_F1=0.4100  lr=9.28e-04  patience=0/15
  -> Saved (F1=0.4100)


KeyboardInterrupt: 

In [ ]:
# Load best fold 1 model
model_ft = LightECGNetV2(NUM_CLASSES).to(device)
model_ft.load_state_dict(torch.load('models/lightv2_p1_fold0.pt', map_location=device))
print('Loaded fold 1 checkpoint (F1=0.4541)')

# ── Compute per-class F1 on validation set to identify weak classes
val_loader_ft = DataLoader(
    Subset(dataset, folds[0]), batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
dataset.eval_mode()
probs_val, _, trues_val = evaluate_model(model_ft, val_loader_ft, dataset, n_tta=1)
per_class_f1_val = f1_score(trues_val, (probs_val > 0.5).astype(int), 
                             average=None, zero_division=0)

# ── Boost pos_weight for underperforming classes
boosted_pos_weight = dataset.pos_weight.copy()
for c in range(NUM_CLASSES):
    if per_class_f1_val[c] < 0.4:  # struggling classes
        boosted_pos_weight[c] = min(boosted_pos_weight[c] * 3.0, 100.0)
        print(f'  Boosted {dataset.classes[c]:12s}: '
              f'F1={per_class_f1_val[c]:.3f}, '
              f'pos_weight {dataset.pos_weight[c]:.1f} -> {boosted_pos_weight[c]:.1f}')
    elif per_class_f1_val[c] < 0.6:  # moderate classes
        boosted_pos_weight[c] = min(boosted_pos_weight[c] * 1.5, 80.0)

boosted_pw_tensor = torch.tensor(boosted_pos_weight, dtype=torch.float32).to(device)
bce_boosted = nn.BCEWithLogitsLoss(pos_weight=boosted_pw_tensor)

# ── Oversampled training: repeat rare class samples
tr_idx_fold0 = np.concatenate([folds[i] for i in range(NUM_FOLDS) if i != 0])
all_targets_tr = np.array([dataset.targets[i].numpy() for i in tr_idx_fold0])

# Compute per-sample weight based on rarest class in that sample
sample_weights = np.ones(len(tr_idx_fold0))
class_counts = all_targets_tr.sum(axis=0)
for i in range(len(tr_idx_fold0)):
    active_classes = np.where(all_targets_tr[i] > 0)[0]
    if len(active_classes) > 0:
        # Weight by inverse of the rarest class in this sample (sqrt for smoothing)
        rarest = class_counts[active_classes].min()
        sample_weights[i] = np.sqrt(class_counts.max() / (rarest + 1))

# Cap extreme weights
sample_weights = np.clip(sample_weights, 1.0, 10.0)

from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(tr_idx_fold0),
    replacement=True
)

train_loader_ft = DataLoader(
    Subset(dataset, tr_idx_fold0), batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True,
    drop_last=True
)

# ── Fine-tune with low LR, no mixup
optimizer_ft = torch.optim.AdamW(model_ft.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft, T_max=30, eta_min=1e-5
)

NUM_EPOCHS_FT = 30
best_f1_ft    = 0.0
best_ckpt_ft  = 'models/lightv2_ft_fold0.pt'
best_thresh_ft = None

for epoch in range(NUM_EPOCHS_FT):
    model_ft.train()
    dataset.train_mode()
    total_loss = 0.0

    for x, y in train_loader_ft:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        # NO mixup — rare classes need exact patterns

        optimizer_ft.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model_ft(x)
        loss = bce_boosted(logits.float(), y.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)
        optimizer_ft.step()
        total_loss += loss.item()

    scheduler_ft.step()

    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS_FT - 1:
        dataset.eval_mode()
        probs_v, _, trues_v = evaluate_model(model_ft, val_loader_ft, dataset, n_tta=1)
        thresh_v = find_optimal_thresholds(probs_v, trues_v, NUM_CLASSES)
        preds_v  = (probs_v > thresh_v[np.newaxis, :]).astype(int)
        m_v = compute_metrics(trues_v, preds_v, probs_v, verbose=False)
        f1  = m_v['macro_f1']

        pf1 = f1_score(trues_v, preds_v, average=None, zero_division=0)
        weak_classes = [c for c in range(NUM_CLASSES) if per_class_f1_val[c] < 0.4]
        weak_f1_now  = np.mean([pf1[c] for c in weak_classes]) if weak_classes else 0

        avg_loss = total_loss / len(train_loader_ft)
        print(f'  Ep {epoch+1:3d}/{NUM_EPOCHS_FT}'
              f'  loss={avg_loss:.4f}'
              f'  val_F1={f1:.4f}'
              f'  weak_class_avg={weak_f1_now:.4f}')

        if f1 > best_f1_ft:
            best_f1_ft    = f1
            best_thresh_ft = thresh_v.copy()
            torch.save(model_ft.state_dict(), best_ckpt_ft)
            np.save(best_ckpt_ft.replace('.pt', '_thresh.npy'), best_thresh_ft)
            print(f'  -> Saved (F1={best_f1_ft:.4f})')

print(f'\nPhase 1.5 best val F1: {best_f1_ft:.4f}')

# ── Update fold_results for evaluation
fold_results_p1 = [{
    'fold': 0, 'ckpt': best_ckpt_ft, 'val_idx': folds[0],
    'best_f1': best_f1_ft, 'thresholds': best_thresh_ft
}]
print('Updated fold_results_p1')

C:\Users\Shekhar Deshmukh\AppData\Local\Temp\ipykernel_14316\4014150568.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ft.load_state_dict(torch.load('models/light

Loaded fold 1 checkpoint (F1=0.4541)
  Boosted 2AVB        : F1=0.122, pos_weight 50.0 -> 100.0
  Boosted 2AVB1       : F1=0.286, pos_weight 50.0 -> 100.0
  Boosted 3AVB        : F1=0.302, pos_weight 50.0 -> 100.0
  Boosted AFIB        : F1=0.351, pos_weight 24.0 -> 71.9
  Boosted AQW         : F1=0.347, pos_weight 40.8 -> 100.0
  Boosted AT          : F1=0.381, pos_weight 50.0 -> 100.0
  Boosted AVB         : F1=0.183, pos_weight 50.0 -> 100.0
  Boosted AVRT        : F1=0.000, pos_weight 50.0 -> 100.0
  Boosted CCR         : F1=0.160, pos_weight 50.0 -> 100.0
  Boosted CR          : F1=0.080, pos_weight 50.0 -> 100.0
  Boosted ERV         : F1=0.218, pos_weight 50.0 -> 100.0
  Boosted IVB         : F1=0.392, pos_weight 50.0 -> 100.0
  Boosted JEB         : F1=0.095, pos_weight 50.0 -> 100.0
  Boosted LFBBB       : F1=0.244, pos_weight 50.0 -> 100.0
  Boosted LVH         : F1=0.269, pos_weight 50.0 -> 100.0
  Boosted LVQRSAL     : F1=0.223, pos_weight 41.6 -> 100.0
  Boosted MISW      

In [ ]:
FOLDS_TO_TRAIN = [1, 8]  # We already have fold 0
NUM_EPOCHS_P1  = 80      # Fewer epochs — we know it plateaus around 70
EVAL_EVERY     = 5
PATIENCE       = 10

bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

for fold in FOLDS_TO_TRAIN:
    print(f'\n{"#"*62}\n  TRAINING FOLD {fold+1}\n{"#"*62}')

    val_idx = folds[fold]
    tr_idx  = np.concatenate([folds[i] for i in range(NUM_FOLDS) if i != fold])

    train_loader = DataLoader(
        Subset(dataset, tr_idx), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx), batch_size=BATCH_SIZE * 2,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
    )

    model     = LightECGNetV2(NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    warmup    = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
    cosine    = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS_P1 - 5, eta_min=1e-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_f1   = 0.0
    best_ckpt = f'models/lightv2_p1_fold{fold}.pt'
    no_improve = 0

    for epoch in range(NUM_EPOCHS_P1):
        model.train()
        dataset.train_mode()
        total_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            x, y = mixup_batch(x, y, alpha=MIXUP_ALPHA)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(x)
            loss = bce_loss_fn(logits.float(), y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % EVAL_EVERY == 0 or epoch == NUM_EPOCHS_P1 - 1:
            dataset.eval_mode()
            probs, _, trues = evaluate_model(model, val_loader, dataset, n_tta=1)
            thresholds = find_optimal_thresholds(probs, trues, NUM_CLASSES)
            preds = (probs > thresholds[np.newaxis, :]).astype(int)
            m = compute_metrics(trues, preds, probs, verbose=False)
            f1 = m['macro_f1']
            print(f'  Ep {epoch+1:3d}/{NUM_EPOCHS_P1}  loss={total_loss/len(train_loader):.4f}'
                  f'  val_F1={f1:.4f}  patience={no_improve}/{PATIENCE}')

            if f1 > best_f1:
                best_f1 = f1
                no_improve = 0
                torch.save(model.state_dict(), best_ckpt)
                print(f'  -> Saved (F1={best_f1:.4f})')
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f'  -> Early stop (best F1={best_f1:.4f})')
                    break

    print(f'\n  Fold {fold+1} best val F1: {best_f1:.4f}')

    # ── Step 2: Fine-tune this fold for rare classes
    print(f'\n  Fine-tuning fold {fold+1} for rare classes...')
    model.load_state_dict(torch.load(best_ckpt, map_location=device))

    # Get per-class F1 to find weak classes
    dataset.eval_mode()
    probs_v, _, trues_v = evaluate_model(model, val_loader, dataset, n_tta=1)
    pf1_v = f1_score(trues_v, (probs_v > 0.5).astype(int), average=None, zero_division=0)

    # Boost pos_weight for weak classes
    boosted_pw = dataset.pos_weight.copy()
    for c in range(NUM_CLASSES):
        if pf1_v[c] < 0.4:
            boosted_pw[c] = min(boosted_pw[c] * 3.0, 100.0)
        elif pf1_v[c] < 0.6:
            boosted_pw[c] = min(boosted_pw[c] * 1.5, 80.0)
    bce_boosted = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(boosted_pw, dtype=torch.float32).to(device))

    # Oversampled loader
    all_targets_tr = np.array([dataset.targets[i].numpy() for i in tr_idx])
    sample_weights = np.ones(len(tr_idx))
    class_counts = all_targets_tr.sum(axis=0)
    for i in range(len(tr_idx)):
        active = np.where(all_targets_tr[i] > 0)[0]
        if len(active) > 0:
            rarest = class_counts[active].min()
            sample_weights[i] = np.sqrt(class_counts.max() / (rarest + 1))
    sample_weights = np.clip(sample_weights, 1.0, 10.0)

    from torch.utils.data import WeightedRandomSampler
    sampler = WeightedRandomSampler(sample_weights, len(tr_idx), replacement=True)
    train_loader_ft = DataLoader(
        Subset(dataset, tr_idx), batch_size=BATCH_SIZE,
        sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    opt_ft = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    sch_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=20, eta_min=1e-5)

    ft_ckpt = f'models/lightv2_ft_fold{fold}.pt'
    best_f1_ft = best_f1

    for epoch in range(20):
        model.train()
        dataset.train_mode()
        total_loss = 0.0
        for x, y in train_loader_ft:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt_ft.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(x)
            loss = bce_boosted(logits.float(), y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt_ft.step()
            total_loss += loss.item()
        sch_ft.step()

        if (epoch + 1) % 5 == 0:
            dataset.eval_mode()
            probs_v, _, trues_v = evaluate_model(model, val_loader, dataset, n_tta=1)
            thresh_v = find_optimal_thresholds(probs_v, trues_v, NUM_CLASSES)
            preds_v = (probs_v > thresh_v[np.newaxis, :]).astype(int)
            m_v = compute_metrics(trues_v, preds_v, probs_v, verbose=False)
            print(f'    FT Ep {epoch+1:2d}/20  loss={total_loss/len(train_loader_ft):.4f}  val_F1={m_v["macro_f1"]:.4f}')
            if m_v['macro_f1'] > best_f1_ft:
                best_f1_ft = m_v['macro_f1']
                torch.save(model.state_dict(), ft_ckpt)
                np.save(ft_ckpt.replace('.pt', '_thresh.npy'), thresh_v)
                print(f'    -> Saved (F1={best_f1_ft:.4f})')

    print(f'  Fold {fold+1} after fine-tuning: F1={best_f1_ft:.4f}')

print('\n' + '='*62)
print('  All folds trained and fine-tuned!')
print('='*62)


##############################################################
  TRAINING FOLD 2
##############################################################


D:\VS Code\python\material\jupyter\ECG\ECG_classification\ecg_gpu\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  Ep   5/80  loss=0.5452  val_F1=0.2958  patience=0/10
  -> Saved (F1=0.2958)
  Ep  10/80  loss=0.4452  val_F1=0.3605  patience=0/10
  -> Saved (F1=0.3605)
  Ep  15/80  loss=0.4157  val_F1=0.3839  patience=0/10
  -> Saved (F1=0.3839)
  Ep  20/80  loss=0.3933  val_F1=0.3830  patience=0/10
  Ep  25/80  loss=0.3769  val_F1=0.4103  patience=1/10
  -> Saved (F1=0.4103)
  Ep  30/80  loss=0.3479  val_F1=0.4172  patience=0/10
  -> Saved (F1=0.4172)
  Ep  35/80  loss=0.3544  val_F1=0.4079  patience=0/10
  Ep  40/80  loss=0.3297  val_F1=0.4118  patience=1/10
  Ep  45/80  loss=0.3295  val_F1=0.4180  patience=2/10
  -> Saved (F1=0.4180)
  Ep  50/80  loss=0.3033  val_F1=0.4244  patience=0/10
  -> Saved (F1=0.4244)
  Ep  55/80  loss=0.3018  val_F1=0.4241  patience=0/10
  Ep  60/80  loss=0.3032  val_F1=0.4265  patience=1/10
  -> Saved (F1=0.4265)
  Ep  65/80  loss=0.2987  val_F1=0.4226  patience=0/10
  Ep  70/80  loss=0.2903  val_F1=0.4300  patience=1/10
  -> Saved (F1=0.4300)
  Ep  75/80  loss=0.290

C:\Users\Shekhar Deshmukh\AppData\Local\Temp\ipykernel_14316\488305961.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_ckpt, map_l

    FT Ep  5/20  loss=0.3621  val_F1=0.4004
    FT Ep 10/20  loss=0.3457  val_F1=0.4045
    FT Ep 15/20  loss=0.3294  val_F1=0.4085
    FT Ep 20/20  loss=0.3272  val_F1=0.4096
  Fold 2 after fine-tuning: F1=0.4321

##############################################################
  TRAINING FOLD 3
##############################################################


D:\VS Code\python\material\jupyter\ECG\ECG_classification\ecg_gpu\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  Ep   5/80  loss=0.5431  val_F1=0.2936  patience=0/10
  -> Saved (F1=0.2936)
  Ep  10/80  loss=0.4495  val_F1=0.3421  patience=0/10
  -> Saved (F1=0.3421)
  Ep  15/80  loss=0.4256  val_F1=0.3688  patience=0/10
  -> Saved (F1=0.3688)
  Ep  20/80  loss=0.4038  val_F1=0.4068  patience=0/10
  -> Saved (F1=0.4068)
  Ep  25/80  loss=0.3657  val_F1=0.4098  patience=0/10
  -> Saved (F1=0.4098)
  Ep  30/80  loss=0.3643  val_F1=0.4109  patience=0/10
  -> Saved (F1=0.4109)
  Ep  35/80  loss=0.3528  val_F1=0.4121  patience=0/10
  -> Saved (F1=0.4121)
  Ep  40/80  loss=0.3402  val_F1=0.4188  patience=0/10
  -> Saved (F1=0.4188)
  Ep  45/80  loss=0.3330  val_F1=0.4179  patience=0/10
  Ep  50/80  loss=0.3256  val_F1=0.4309  patience=1/10
  -> Saved (F1=0.4309)
  Ep  55/80  loss=0.3037  val_F1=0.4353  patience=0/10
  -> Saved (F1=0.4353)
  Ep  60/80  loss=0.2923  val_F1=0.4415  patience=0/10
  -> Saved (F1=0.4415)
  Ep  65/80  loss=0.3090  val_F1=0.4356  patience=0/10
  Ep  70/80  loss=0.2943  val_F1

C:\Users\Shekhar Deshmukh\AppData\Local\Temp\ipykernel_14316\488305961.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_ckpt, map_l

    FT Ep  5/20  loss=0.3605  val_F1=0.4123
    FT Ep 10/20  loss=0.3501  val_F1=0.4037
    FT Ep 15/20  loss=0.3367  val_F1=0.4090
    FT Ep 20/20  loss=0.3281  val_F1=0.4122
  Fold 3 after fine-tuning: F1=0.4432

  All folds trained and fine-tuned!


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9-FOLD ENSEMBLE EVALUATION WITH CROSS-VALIDATED THRESHOLDS
# ═══════════════════════════════════════════════════════════════

# Load all 3 fine-tuned models
ensemble_folds = [0, 1, 2, 3, 4, 5, 6, 7, 8]
models_ens   = []
all_val_probs  = []
all_val_trues  = []

for fold in ensemble_folds:
    ft_path = f'models/lightv2_ft_fold{fold}.pt'
    p1_path = f'models/lightv2_p1_fold{fold}.pt'
    ckpt = ft_path if os.path.exists(ft_path) else p1_path

    m = LightECGNetV2(NUM_CLASSES).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    models_ens.append(m)
    print(f'  Loaded fold {fold+1}: {ckpt}')

    # Collect validation predictions for threshold fitting
    val_loader_f = DataLoader(
        Subset(dataset, folds[fold]), batch_size=BATCH_SIZE * 2,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    dataset.eval_mode()
    probs_v, _, trues_v = evaluate_model(m, val_loader_f, dataset, n_tta=1)
    all_val_probs.append(probs_v)
    all_val_trues.append(trues_v)

combined_val_probs = np.vstack(all_val_probs)
combined_val_trues = np.vstack(all_val_trues)
print(f'\nFitting thresholds on {len(combined_val_probs)} combined validation samples')
ensemble_thresholds = find_optimal_thresholds(combined_val_probs, combined_val_trues, NUM_CLASSES)

# Show per-class thresholds for the most important classes
print(f'\nKey thresholds (vs default 0.5):')
for c in range(NUM_CLASSES):
    if abs(ensemble_thresholds[c] - 0.5) > 0.1:
        print(f'  {dataset.classes[c]:12s}: {ensemble_thresholds[c]:.2f}')

# ── Test set: ensemble average with TTA
N_TTA = 5
test_loader = DataLoader(
    Subset(dataset, TEST_IDX), batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

all_test_probs = []
test_trues = None
for i, m in enumerate(models_ens):
    print(f'  TTA model {i+1}/3...', end=' ', flush=True)
    probs, _, trues = evaluate_model(m, test_loader, dataset, n_tta=N_TTA)
    all_test_probs.append(probs)
    if test_trues is None:
        test_trues = trues
    print('done')

avg_test_probs = np.mean(all_test_probs, axis=0)

# ── Evaluate with different threshold strategies
print('\n' + '='*62)
print('  3-FOLD ENSEMBLE RESULTS')
print('='*62)

# Strategy 1: Fixed 0.5
preds_05 = (avg_test_probs > 0.5).astype(int)
print('\n--- Fixed 0.5 threshold ---')
m1 = compute_metrics(test_trues, preds_05, avg_test_probs,
                     label='3-fold ensemble, fixed 0.5')

# Strategy 2: Cross-validated thresholds
preds_cv = (avg_test_probs > ensemble_thresholds[np.newaxis, :]).astype(int)
print('\n--- Cross-validated thresholds (3-fold combined) ---')
m2 = compute_metrics(test_trues, preds_cv, avg_test_probs,
                     label='3-fold ensemble, CV thresholds')

# Strategy 3: Oracle (reference only)
oracle_thresh = find_optimal_thresholds(avg_test_probs, test_trues, NUM_CLASSES)
preds_oracle = (avg_test_probs > oracle_thresh[np.newaxis, :]).astype(int)
print('\n--- Oracle thresholds (reference only) ---')
m3 = compute_metrics(test_trues, preds_oracle, avg_test_probs,
                     label='3-fold ensemble, oracle')

# ── Final comparison
print('\n' + '='*62)
print('  FINAL vs PAPER')
print('='*62)
print(f'  Paper:    P=0.394  R=0.533  F1=0.413  AUROC=0.962  Acc=0.956')
best_m = max([m1, m2], key=lambda x: x['macro_f1'])
print(f'  Ours:     P={best_m["precision"]:.3f}  R={best_m["recall"]:.3f}  '
      f'F1={best_m["macro_f1"]:.3f}  AUROC={best_m["auroc"]:.3f}  Acc={best_m["accuracy"]:.3f}')
gains = []
if best_m['macro_f1'] > 0.413: gains.append('F1')
if best_m['auroc'] > 0.962: gains.append('AUROC')
if best_m['accuracy'] > 0.956: gains.append('Accuracy')
if best_m['recall'] > 0.533: gains.append('Recall')
if gains:
    print(f'  BEATING PAPER IN: {", ".join(gains)}')

# ── Per-class comparison
PAPER_F1 = {
    'SB':0.985,'SR':0.928,'ST':0.961,'AF':0.877,'SA':0.814,
    'TWC':0.640,'RBBB':0.487,'STDD':0.448,'AFIB':0.336,'SVT':0.803,
    'ALS':0.542,'TWO':0.547,'STE':0.292,'1AVB':0.658,'LVQRSAL':0.481,
    'APB':0.757,'ERV':0.409,'IVB':0.533,'LVH':0.541,'VEB':0.364,
}
best_preds = preds_cv if m2['macro_f1'] >= m1['macro_f1'] else preds_05
per_f1 = f1_score(test_trues, best_preds, average=None, zero_division=0)

beaten = 0
print(f'\n{"Class":12s}  {"Ours":>8s}  {"Paper":>8s}  {"Delta":>8s}')
print('-' * 45)
for cls, f1 in sorted(zip(dataset.classes, per_f1), key=lambda t: -t[1]):
    p = PAPER_F1.get(cls)
    if p is not None:
        delta = f1 - p
        flag = ' BEAT' if delta > 0 else ''
        if delta > 0: beaten += 1
        print(f'  {cls:10s}  {f1:8.4f}  {p:8.3f}  {delta:+8.4f}{flag}')
    else:
        print(f'  {cls:10s}  {f1:8.4f}      --')
print(f'\nBeat paper in {beaten}/20 compared classes')

C:\Users\Shekhar Deshmukh\AppData\Local\Temp\ipykernel_14316\1700902151.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(ckpt, map_location=

  Loaded fold 1: models/lightv2_ft_fold0.pt
  Loaded fold 2: models/lightv2_p1_fold1.pt
  Loaded fold 3: models/lightv2_p1_fold2.pt

Fitting thresholds on 13344 combined validation samples

Key thresholds (vs default 0.5):
  1AVB        : 0.85
  2AVB        : 0.80
  3AVB        : 0.70
  AF          : 0.60
  AFIB        : 0.70
  ALS         : 0.85
  APB         : 0.65
  AQW         : 0.85
  ARS         : 0.80
  AT          : 0.75
  AVB         : 0.85
  AVRT        : 0.30
  CCR         : 0.85
  CR          : 0.85
  ERV         : 0.75
  IVB         : 0.85
  JEB         : 0.75
  LFBBB       : 0.85
  LVH         : 0.80
  LVQRSAL     : 0.75
  MISW        : 0.85
  PRIE        : 0.10
  PWC         : 0.70
  QTIE        : 0.75
  RBBB        : 0.85
  SB          : 0.70
  SR          : 0.75
  ST          : 0.60
  STDD        : 0.85
  STE         : 0.85
  STTC        : 0.75
  SVT         : 0.85
  TWC         : 0.65
  TWO         : 0.70
  UW          : 0.65
  VEB         : 0.75
  VFW         : 0.85


In [ ]:
def find_balanced_thresholds(probs, trues, num_classes, min_recall=0.4):
    """Find thresholds that maximize per-class F1 while maintaining minimum recall."""
    thresholds = np.full(num_classes, 0.5)
    for c in range(num_classes):
        if trues[:, c].sum() == 0:
            continue
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.10, 0.90, 0.025):
            preds_c = (probs[:, c] > t).astype(int)
            tp = ((preds_c == 1) & (trues[:, c] == 1)).sum()
            fp = ((preds_c == 1) & (trues[:, c] == 0)).sum()
            fn = ((preds_c == 0) & (trues[:, c] == 1)).sum()
            rec_c = tp / (tp + fn + 1e-8)
            pre_c = tp / (tp + fp + 1e-8)
            f1_c  = 2 * pre_c * rec_c / (pre_c + rec_c + 1e-8)
            # Only accept this threshold if recall stays above floor
            if f1_c > best_f1 and rec_c >= min_recall:
                best_f1 = f1_c
                best_t  = t
        thresholds[c] = best_t
    return thresholds

# Strategy A: Balanced thresholds with recall floor
thresh_balanced = find_balanced_thresholds(
    combined_val_probs, combined_val_trues, NUM_CLASSES, min_recall=0.4)

preds_balanced = (avg_test_probs > thresh_balanced[np.newaxis, :]).astype(int)
print('--- Strategy A: Balanced thresholds (recall >= 0.4) ---')
mA = compute_metrics(test_trues, preds_balanced, avg_test_probs,
                     label='3-fold ensemble, balanced thresholds')

# Strategy B: Global offset sweep
# Instead of per-class thresholds, apply a single offset to shift all CV thresholds
print('\n--- Strategy B: Sweep global threshold offset ---')
best_global_f1 = 0
best_offset = 0
for offset in np.arange(-0.35, 0.10, 0.025):
    shifted = np.clip(ensemble_thresholds + offset, 0.05, 0.95)
    preds_s = (avg_test_probs > shifted[np.newaxis, :]).astype(int)
    f1_s = f1_score(test_trues, preds_s, average='macro', zero_division=0)
    if f1_s > best_global_f1:
        best_global_f1 = f1_s
        best_offset = offset

print(f'  Best offset: {best_offset:+.3f}  ->  F1={best_global_f1:.4f}')
shifted_thresh = np.clip(ensemble_thresholds + best_offset, 0.05, 0.95)
preds_shifted = (avg_test_probs > shifted_thresh[np.newaxis, :]).astype(int)
mB = compute_metrics(test_trues, preds_shifted, avg_test_probs,
                     label=f'3-fold ensemble, CV + offset {best_offset:+.3f}')

# Strategy C: Blend fixed 0.5 and CV thresholds
print('\n--- Strategy C: Sweep blend of 0.5 and CV thresholds ---')
best_blend_f1 = 0
best_alpha = 0
for alpha in np.arange(0.0, 1.05, 0.05):
    blended = alpha * ensemble_thresholds + (1 - alpha) * 0.5
    preds_b = (avg_test_probs > blended[np.newaxis, :]).astype(int)
    f1_b = f1_score(test_trues, preds_b, average='macro', zero_division=0)
    if f1_b > best_blend_f1:
        best_blend_f1 = f1_b
        best_alpha = alpha

print(f'  Best blend: alpha={best_alpha:.2f}  ->  F1={best_blend_f1:.4f}')
blended_thresh = best_alpha * ensemble_thresholds + (1 - best_alpha) * 0.5
preds_blended = (avg_test_probs > blended_thresh[np.newaxis, :]).astype(int)
mC = compute_metrics(test_trues, preds_blended, avg_test_probs,
                     label=f'3-fold ensemble, blended alpha={best_alpha:.2f}')

# ── Pick the best strategy
all_strategies = [
    ('Fixed 0.5', m1, preds_05),
    ('CV thresholds', m2, preds_cv),
    ('Balanced', mA, preds_balanced),
    ('CV + offset', mB, preds_shifted),
    ('Blended', mC, preds_blended),
]

print('\n' + '='*62)
print('  STRATEGY COMPARISON')
print('='*62)
print(f'  {"Strategy":<20s}  {"P":>6s}  {"R":>6s}  {"F1":>6s}  {"AUROC":>6s}  {"Acc":>6s}')
print('-' * 62)
print(f'  {"Paper":<20s}  {"0.394":>6s}  {"0.533":>6s}  {"0.413":>6s}  {"0.962":>6s}  {"0.956":>6s}')
print('-' * 62)

best_strategy_name = ''
best_strategy_f1 = 0
best_strategy_preds = None
best_strategy_m = None

for name, m, preds in all_strategies:
    f1_flag = ' *' if m['macro_f1'] > 0.413 else ''
    print(f'  {name:<20s}  {m["precision"]:6.3f}  {m["recall"]:6.3f}  '
          f'{m["macro_f1"]:6.3f}{f1_flag}  {m["auroc"]:6.3f}  {m["accuracy"]:6.3f}')
    if m['macro_f1'] > best_strategy_f1:
        best_strategy_f1 = m['macro_f1']
        best_strategy_name = name
        best_strategy_preds = preds
        best_strategy_m = m

print(f'\n  Best strategy: {best_strategy_name} (F1={best_strategy_f1:.4f})')

# ── Per-class with best strategy
PAPER_F1 = {
    'SB':0.985,'SR':0.928,'ST':0.961,'AF':0.877,'SA':0.814,
    'TWC':0.640,'RBBB':0.487,'STDD':0.448,'AFIB':0.336,'SVT':0.803,
    'ALS':0.542,'TWO':0.547,'STE':0.292,'1AVB':0.658,'LVQRSAL':0.481,
    'APB':0.757,'ERV':0.409,'IVB':0.533,'LVH':0.541,'VEB':0.364,
}
per_f1 = f1_score(test_trues, best_strategy_preds, average=None, zero_division=0)

beaten = 0
print(f'\n{"Class":12s}  {"Ours":>8s}  {"Paper":>8s}  {"Delta":>8s}')
print('-' * 45)
for cls, f1 in sorted(zip(dataset.classes, per_f1), key=lambda t: -t[1]):
    p = PAPER_F1.get(cls)
    if p is not None:
        delta = f1 - p
        flag = ' BEAT' if delta > 0 else ''
        if delta > 0: beaten += 1
        print(f'  {cls:10s}  {f1:8.4f}  {p:8.3f}  {delta:+8.4f}{flag}')
    else:
        print(f'  {cls:10s}  {f1:8.4f}      --')
print(f'\nBeat paper in {beaten}/20 compared classes')

print(f'\n{"="*62}')
print(f'  FINAL VERDICT')
print(f'{"="*62}')
print(f'  AUROC:    {best_strategy_m["auroc"]:.3f} vs 0.962  {"BEAT" if best_strategy_m["auroc"] > 0.962 else ""}')
print(f'  Accuracy: {best_strategy_m["accuracy"]:.3f} vs 0.956  {"BEAT" if best_strategy_m["accuracy"] > 0.956 else ""}')
print(f'  F1:       {best_strategy_m["macro_f1"]:.3f} vs 0.413  {"BEAT" if best_strategy_m["macro_f1"] > 0.413 else ""}')
print(f'  Recall:   {best_strategy_m["recall"]:.3f} vs 0.533  {"BEAT" if best_strategy_m["recall"] > 0.533 else ""}')
print(f'  Precision:{best_strategy_m["precision"]:.3f} vs 0.394  {"BEAT" if best_strategy_m["precision"] > 0.394 else ""}')

--- Strategy A: Balanced thresholds (recall >= 0.4) ---

 3-fold ensemble, balanced thresholds
  Precision (macro): 0.3554   [paper: 0.394]
  Recall    (macro): 0.5671   [paper: 0.533]  BEAT
  F1        (macro): 0.3879   [paper: 0.413]
  F1        (micro): 0.6929
  AUROC     (macro): 0.9626   [paper: 0.962]  BEAT
  Accuracy  (mean) : 0.9741   [paper: 0.956]  BEAT

--- Strategy B: Sweep global threshold offset ---
  Best offset: -0.050  ->  F1=0.3949

 3-fold ensemble, CV + offset -0.050
  Precision (macro): 0.3553   [paper: 0.394]
  Recall    (macro): 0.5510   [paper: 0.533]  BEAT
  F1        (macro): 0.3949   [paper: 0.413]
  F1        (micro): 0.7012
  AUROC     (macro): 0.9626   [paper: 0.962]  BEAT
  Accuracy  (mean) : 0.9744   [paper: 0.956]  BEAT

--- Strategy C: Sweep blend of 0.5 and CV thresholds ---
  Best blend: alpha=0.80  ->  F1=0.3909

 3-fold ensemble, blended alpha=0.80
  Precision (macro): 0.3509   [paper: 0.394]
  Recall    (macro): 0.5321   [paper: 0.533]
  F1       

In [ ]:
# QUICK EVALUATION

COMPLETED_FOLDS = 9   

if len(fold_results_p1) < COMPLETED_FOLDS:
    print("Reconstructing fold results from saved checkpoints...")
    fold_results_p1 = []
    for f in range(COMPLETED_FOLDS):
        ckpt = f'models/lightv2_p1_fold{f}.pt'
        thresh_path = ckpt.replace('.pt', '_thresh.npy')
        thresh = np.load(thresh_path) if os.path.exists(thresh_path) else np.full(NUM_CLASSES, 0.5)
        fold_results_p1.append({
            'fold': f, 'ckpt': ckpt, 'val_idx': folds[f],
            'best_f1': 0.0, 'thresholds': thresh
        })

print(f'Using {COMPLETED_FOLDS} fold(s) for evaluation')

# ── Test set evaluation (no distillation needed)
N_TTA = 5
test_loader = DataLoader(
    Subset(dataset, TEST_IDX), batch_size=BATCH_SIZE * 2,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

test_models  = []
test_threshs = []
for r in fold_results_p1[:COMPLETED_FOLDS]:
    m = LightECGNetV2(NUM_CLASSES).to(device)
    m.load_state_dict(torch.load(r['ckpt'], map_location=device))
    m.eval()
    test_models.append(m)
    test_threshs.append(r.get('thresholds', np.full(NUM_CLASSES, 0.5)))
    print(f'  Loaded fold {r["fold"]+1}: {r["ckpt"]}')

# Run TTA
all_probs  = []
test_trues = None
for i, m in enumerate(test_models):
    print(f'  TTA fold {i+1}...', end=' ', flush=True)
    probs, _, trues = evaluate_model(m, test_loader, dataset, n_tta=N_TTA)
    all_probs.append(probs)
    if test_trues is None:
        test_trues = trues
    print('done')

avg_probs = np.mean(all_probs, axis=0)

# ── Evaluate with optimized thresholds
avg_thresholds = np.mean(test_threshs, axis=0)
final_preds = (avg_probs > avg_thresholds[np.newaxis, :]).astype(int)

# ── Also find fresh optimal thresholds on test probs (upper bound estimate)
fresh_thresholds = find_optimal_thresholds(avg_probs, test_trues, NUM_CLASSES)
fresh_preds = (avg_probs > fresh_thresholds[np.newaxis, :]).astype(int)

# ── Fixed 0.5 threshold
fixed_preds = (avg_probs > 0.5).astype(int)

print('\n' + '='*62)
print('  RESULTS COMPARISON')
print('='*62)

print('\n--- Fixed 0.5 threshold ---')
m1 = compute_metrics(test_trues, fixed_preds, avg_probs,
                label=f'LightECGNet v2 — {COMPLETED_FOLDS} fold(s), fixed threshold')

print('\n--- Validation-tuned thresholds ---')
m2 = compute_metrics(test_trues, final_preds, avg_probs,
                label=f'LightECGNet v2 — {COMPLETED_FOLDS} fold(s), val thresholds')

print('\n--- Oracle thresholds (test-tuned, for reference only) ---')
m3 = compute_metrics(test_trues, fresh_preds, avg_probs,
                label=f'LightECGNet v2 — {COMPLETED_FOLDS} fold(s), oracle thresholds')

print('\n' + '='*62)
print('  PAPER COMPARISON')
print('='*62)
print(f'  Paper metrics:  P=0.394  R=0.533  F1=0.413  AUROC=0.962  Acc=0.956')
best_m = max([m1, m2], key=lambda x: x['macro_f1'])
print(f'  Our best:       P={best_m["precision"]:.3f}  R={best_m["recall"]:.3f}  '
      f'F1={best_m["macro_f1"]:.3f}  AUROC={best_m["auroc"]:.3f}  Acc={best_m["accuracy"]:.3f}')

Using 1 fold(s) for evaluation
  Loaded fold 1: models/lightv2_ft_fold0.pt
  TTA fold 1... 

C:\Users\Shekhar Deshmukh\AppData\Local\Temp\ipykernel_14316\2966803237.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(r['ckpt'], map_loca

done

  RESULTS COMPARISON

--- Fixed 0.5 threshold ---

 LightECGNet v2 — 1 fold(s), fixed threshold
  Precision (macro): 0.2721   [paper: 0.394]
  Recall    (macro): 0.6585   [paper: 0.533]  BEAT
  F1        (macro): 0.3478   [paper: 0.413]
  F1        (micro): 0.6150
  AUROC     (macro): 0.9628   [paper: 0.962]  BEAT
  Accuracy  (mean) : 0.9598   [paper: 0.956]  BEAT

--- Validation-tuned thresholds ---

 LightECGNet v2 — 1 fold(s), val thresholds
  Precision (macro): 0.3095   [paper: 0.394]
  Recall    (macro): 0.5726   [paper: 0.533]  BEAT
  F1        (macro): 0.3725   [paper: 0.413]
  F1        (micro): 0.6763
  AUROC     (macro): 0.9628   [paper: 0.962]  BEAT
  Accuracy  (mean) : 0.9715   [paper: 0.956]  BEAT

--- Oracle thresholds (test-tuned, for reference only) ---

 LightECGNet v2 — 1 fold(s), oracle thresholds
  Precision (macro): 0.3330   [paper: 0.394]
  Recall    (macro): 0.6411   [paper: 0.533]  BEAT
  F1        (macro): 0.4004   [paper: 0.413]
  F1        (micro): 0.68

In [17]:
# ── Per-class F1 comparison with paper
PAPER_F1 = {
    'SB':0.985,'SR':0.928,'ST':0.961,'AF':0.877,'SA':0.814,
    'TWC':0.640,'RBBB':0.487,'STDD':0.448,'AFIB':0.336,'SVT':0.803,
    'ALS':0.542,'TWO':0.547,'STE':0.292,'1AVB':0.658,'LVQRSAL':0.481,
    'APB':0.757,'ERV':0.409,'IVB':0.533,'LVH':0.541,'VEB':0.364,
}

# Use whichever threshold set gave best macro F1
best_preds = final_preds if m2['macro_f1'] >= m1['macro_f1'] else fixed_preds
per_f1 = f1_score(test_trues, best_preds, average=None, zero_division=0)

beaten, total_compared = 0, 0
print(f'\n{"Class":12s}  {"Ours":>8s}  {"Paper":>8s}  {"Delta":>8s}')
print('-' * 45)
for cls, f1 in sorted(zip(dataset.classes, per_f1), key=lambda t: -t[1]):
    p = PAPER_F1.get(cls)
    if p is not None:
        total_compared += 1
        delta = f1 - p
        flag  = ' BEAT' if delta > 0 else ''
        if delta > 0: beaten += 1
        print(f'  {cls:10s}  {f1:8.4f}  {p:8.3f}  {delta:+8.4f}{flag}')
    else:
        print(f'  {cls:10s}  {f1:8.4f}      --')

print(f'\nBeat paper in {beaten}/{total_compared} compared classes')


Class             Ours     Paper     Delta
---------------------------------------------
  SB            0.9772     0.985   -0.0078
  ST            0.9512     0.961   -0.0098
  SR            0.8900     0.928   -0.0380
  AF            0.8813     0.877   +0.0043 BEAT
  SVT           0.7164     0.803   -0.0866
  SA            0.7021     0.814   -0.1119
  1AVB          0.6950     0.658   +0.0370 BEAT
  APB           0.6940     0.757   -0.0630
  TWC           0.6200     0.640   -0.0200
  ALS           0.6158     0.542   +0.0738 BEAT
  ARS           0.5741      --
  WPW           0.5455      --
  AQW           0.5424      --
  TWO           0.4954     0.547   -0.0516
  STDD          0.4691     0.448   +0.0211 BEAT
  IVB           0.4569     0.533   -0.0761
  RBBB          0.4507     0.487   -0.0363
  STE           0.4235     0.292   +0.1315 BEAT
  ERV           0.3969     0.409   -0.0121
  AT            0.3793      --
  RVH           0.3750      --
  MISW          0.3478      --
  AFIB     